# Kuro Siwo — tải raw
Chạy lần lượt các cell. Sửa đường dẫn Drive ở dưới. Raw ~7,53 GB; cần đủ dung lượng trên Drive và local. Chỉ tải raw, không tạo dữ liệu processed. Các hàm tải/kiểm checksum nằm trực tiếp trong notebook.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/kuro_siwo")
LOCAL_ROOT = Path("/content/kuro_siwo")
RAW_ON_DRIVE = DRIVE_ROOT / "raw" / "geobench_kuro_siwo.tortilla"
REPORT_ROOT = DRIVE_ROOT / "reports"

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *["huggingface_hub>=0.27,<3"]], check=True
)
from google.colab import drive  # noqa: E402

drive.mount("/content/drive")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

## Hàm tải và kiểm tra tính toàn vẹn

In [ ]:
import hashlib
import os
import shutil
from pathlib import Path

REPO = "aialliance/kuro_siwo"
REVISION = "25997dc683df721de44fe6f68a4ce1b36f4562eb"
RAW_NAME = "geobench_kuro_siwo.tortilla"
RAW_BYTES = 7529588233
RAW_SHA256 = "4830fe6f23bf9750dee0c765850724b55026bf5d47cb67162d3ef7dcb04c3bbd"

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
def verified_copy(source, destination, expected_hash=None):
    """Never trust a partial copy or overwrite an unrelated existing artifact."""
    source, destination = Path(source), Path(destination)
    expected_hash = expected_hash or sha256(source)
    if destination.exists():
        if sha256(destination) != expected_hash:
            raise ValueError(f"Checksum mismatch: {destination}; move it aside before retry")
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_name(destination.name + ".partial")
    shutil.copyfile(source, tmp)
    if sha256(tmp) != expected_hash:
        raise ValueError(f"Copy verification failed: {tmp}")
    os.replace(tmp, destination)
    return destination

In [ ]:
def verify_raw(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size != RAW_BYTES:
        raise ValueError(f"Missing/truncated raw archive: {path}")
    digest = sha256(path)
    if digest != RAW_SHA256:
        raise ValueError(f"Raw SHA256 mismatch: {path}; keep aside and retry download")
    return digest

In [ ]:
def download_raw(local_dir, drive_dir=None):
    """Download locally, then optionally back up raw to Drive. Never save processed data."""
    local = Path(local_dir) / RAW_NAME
    backup = Path(drive_dir) / RAW_NAME if drive_dir is not None else None
    if local.exists():
        verify_raw(local)
    elif backup is not None and backup.exists():
        stage_raw(backup, local)
    else:
        from huggingface_hub import hf_hub_download

        local.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading pinned raw (~7.53 GB); Hub can resume partial downloads", flush=True)
        hf_hub_download(
            REPO, RAW_NAME, repo_type="dataset", revision=REVISION, local_dir=str(local.parent)
        )
        verify_raw(local)
    if backup is not None:
        print("Copy/check raw backup on Drive", flush=True)
        verified_copy(local, backup, RAW_SHA256)
    receipt = {
        "dataset": REPO,
        "revision": REVISION,
        "file": RAW_NAME,
        "size_bytes": RAW_BYTES,
        "sha256": RAW_SHA256,
    }
    for directory in {local.parent, backup.parent if backup is not None else local.parent}:
        (directory / "raw_provenance.json").write_text(json.dumps(receipt, indent=2))
    return local

In [ ]:
def stage_raw(source, destination):
    """Model-team staging: missing source raises; no automatic Hub fallback."""
    source, destination = Path(source), Path(destination)
    if not source.is_file() or source.stat().st_size != RAW_BYTES:
        raise ValueError(f"Missing/truncated shared raw archive: {source}")
    if source.resolve() == destination.resolve():
        verify_raw(source)
        return source
    return verified_copy(source, destination, RAW_SHA256)

## Chạy tải và lưu Drive

In [ ]:
if RAW_ON_DRIVE.name != RAW_NAME:
    raise ValueError(f"Giữ tên raw: {RAW_NAME}")
RAW_LOCAL = download_raw(LOCAL_ROOT / "raw", RAW_ON_DRIVE.parent)
print("RAW LOCAL:", RAW_LOCAL)
print("RAW DRIVE:", RAW_ON_DRIVE)
print("SHA256 đã xác minh. Tiếp theo chạy kuro_preprocess.ipynb.")